In [1]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv(r'C:\ds_projects\actionada\Task_1\Daily Household Transactions.csv')
df.head()

,Date,Mode,Category,Subcategory,Note,Amount,Income/Expense,Currency
0,20/09/2018 12:04:08,Cash,Transportation,Train,2 Place 5 to Place 0,30.0,Expense,INR
1,20/09/2018 12:03:15,Cash,Food,snacks,Idli medu Vada mix 2 plates,60.0,Expense,INR
2,19/09/2018,Saving Bank account 1,subscription,Netflix,1 month subscription,199.0,Expense,INR
3,17/09/2018 23:41:17,Saving Bank account 1,subscription,Mobile Service Provider,Data booster pack,19.0,Expense,INR
4,16/09/2018 17:15:08,Cash,Festivals,Ganesh Pujan,Ganesh idol,251.0,Expense,INR


In [3]:
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [4]:
df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True)
display(df.head())
df.dtypes

,Date,Mode,Category,Subcategory,Note,Amount,Income/Expense,Currency
0,2018-09-20 12:04:08,Cash,Transportation,Train,2 Place 5 to Place 0,30.0,Expense,INR
1,2018-09-20 12:03:15,Cash,Food,snacks,Idli medu Vada mix 2 plates,60.0,Expense,INR
2,2018-09-19 00:00:00,Saving Bank account 1,subscription,Netflix,1 month subscription,199.0,Expense,INR
3,2018-09-17 23:41:17,Saving Bank account 1,subscription,Mobile Service Provider,Data booster pack,19.0,Expense,INR
4,2018-09-16 17:15:08,Cash,Festivals,Ganesh Pujan,Ganesh idol,251.0,Expense,INR


Date              datetime64[us]
Mode                         str
Category                     str
Subcategory                  str
Note                         str
Amount                   float64
Income/Expense               str
Currency                     str
dtype: object

In [5]:
df = df.fillna('UNKNOWN')
df.isna().sum()

Date              0
Mode              0
Category          0
Subcategory       0
Note              0
Amount            0
Income/Expense    0
Currency          0
dtype: int64

In [6]:
rare_values = df['Category'].value_counts()[df['Category'].value_counts() < 10].index
df.loc[df['Category'].isin(rare_values), 'Category'] = 'Other'
df['Category'].value_counts()

Category
Food                         906
Transportation               307
Other                        206
Household                    176
subscription                 143
Investment                   101
Health                        94
Family                        71
Apparel                       47
Salary                        43
Money transfer                43
Recurring Deposit             41
Gift                          30
Public Provident Fund         29
Equity Mutual Fund E          22
Beauty                        22
Gpay Reward                   21
Education                     18
Saving Bank account 1         17
maid                          17
Festivals                     16
Equity Mutual Fund A          14
Equity Mutual Fund F          13
Dividend earned on Shares     12
Interest                      12
Culture                       11
Small Cap fund 2              10
Small cap fund 1              10
Name: count, dtype: int64

In [7]:
y = df['Category']
X = df.drop(['Category', 'Subcategory', 'Currency', 'Note'], axis=1)
X.head()

,Date,Mode,Amount,Income/Expense
0,2018-09-20 12:04:08,Cash,30.0,Expense
1,2018-09-20 12:03:15,Cash,60.0,Expense
2,2018-09-19 00:00:00,Saving Bank account 1,199.0,Expense
3,2018-09-17 23:41:17,Saving Bank account 1,19.0,Expense
4,2018-09-16 17:15:08,Cash,251.0,Expense


In [8]:
X['dayofweek'] = X['Date'].dt.dayofweek
X['month'] = X['Date'].dt.month
X['is_weekend'] = (X['Date'].dt.dayofweek >= 5).astype(int)
X.head()

,Date,Mode,Amount,Income/Expense,dayofweek,month,is_weekend
0,2018-09-20 12:04:08,Cash,30.0,Expense,3,9,0
1,2018-09-20 12:03:15,Cash,60.0,Expense,3,9,0
2,2018-09-19 00:00:00,Saving Bank account 1,199.0,Expense,2,9,0
3,2018-09-17 23:41:17,Saving Bank account 1,19.0,Expense,0,9,0
4,2018-09-16 17:15:08,Cash,251.0,Expense,6,9,1


In [9]:
X = X.drop(['Date'], axis=1)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

In [11]:
print(X_train['Mode'].unique())
X_test['Mode'].unique()

<StringArray>
[                 'Cash', 'Saving Bank account 1',           'Credit Card',
  'Share Market Trading',  'Equity Mutual Fund B', 'Saving Bank account 2',
     'Recurring Deposit',            'Debit Card',  'Equity Mutual Fund C',
  'Equity Mutual Fund D']
Length: 10, dtype: str


<StringArray>
[                 'Cash', 'Saving Bank account 1',           'Credit Card',
         'Fixed Deposit',  'Equity Mutual Fund B', 'Saving Bank account 2',
  'Equity Mutual Fund A',            'Debit Card']
Length: 8, dtype: str

In [12]:
obj = ['Mode', 'Income/Expense']
encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train[obj])
X_train[encoder.get_feature_names_out()] = encoder.transform(X_train[obj])
X_train = X_train.drop(obj, axis=1)
X_test[encoder.get_feature_names_out()] = encoder.transform(X_test[obj])
X_test = X_test.drop(obj, axis=1)

C:\ds_projects\ds_env\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [13]:
num = ['Amount']
scaler = StandardScaler()
scaler.fit(X_train[num])
X_train[num] = scaler.transform(X_train[num])
X_test[num] = scaler.transform(X_test[num])

In [14]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

In [15]:
num_classes = len(le.classes_)

In [16]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 1839 entries, 52 to 2031
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Amount                       1839 non-null   float64
 1   dayofweek                    1839 non-null   int32  
 2   month                        1839 non-null   int32  
 3   is_weekend                   1839 non-null   int64  
 4   Mode_Credit Card             1839 non-null   float64
 5   Mode_Debit Card              1839 non-null   float64
 6   Mode_Equity Mutual Fund B    1839 non-null   float64
 7   Mode_Equity Mutual Fund C    1839 non-null   float64
 8   Mode_Equity Mutual Fund D    1839 non-null   float64
 9   Mode_Recurring Deposit       1839 non-null   float64
 10  Mode_Saving Bank account 1   1839 non-null   float64
 11  Mode_Saving Bank account 2   1839 non-null   float64
 12  Mode_Share Market Trading    1839 non-null   float64
 13  Income/Expense_Income        1839

In [17]:
X_test.info()

<class 'pandas.DataFrame'>
Index: 613 entries, 38 to 1592
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Amount                       613 non-null    float64
 1   dayofweek                    613 non-null    int32  
 2   month                        613 non-null    int32  
 3   is_weekend                   613 non-null    int64  
 4   Mode_Credit Card             613 non-null    float64
 5   Mode_Debit Card              613 non-null    float64
 6   Mode_Equity Mutual Fund B    613 non-null    float64
 7   Mode_Equity Mutual Fund C    613 non-null    float64
 8   Mode_Equity Mutual Fund D    613 non-null    float64
 9   Mode_Recurring Deposit       613 non-null    float64
 10  Mode_Saving Bank account 1   613 non-null    float64
 11  Mode_Saving Bank account 2   613 non-null    float64
 12  Mode_Share Market Trading    613 non-null    float64
 13  Income/Expense_Income        613 n

In [18]:
X_test.head()

,Amount,dayofweek,month,is_weekend,Mode_Credit Card,Mode_Debit Card,Mode_Equity Mutual Fund B,Mode_Equity Mutual Fund C,Mode_Equity Mutual Fund D,Mode_Recurring Deposit,Mode_Saving Bank account 1,Mode_Saving Bank account 2,Mode_Share Market Trading,Income/Expense_Income,Income/Expense_Transfer-Out
38,-0.215851,2,8,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
28,0.762013,5,9,1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
476,-0.110741,0,3,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1458,-0.138703,4,4,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
921,-0.138703,6,10,1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [19]:
X_train = torch.tensor(X_train.values, dtype=torch.float32)
y_train_encoded = torch.tensor(y_train_encoded, dtype=torch.long) 
X_test = torch.tensor(X_test.values, dtype=torch.float32)
y_test_encoded = torch.tensor(y_test_encoded, dtype=torch.long)  

In [29]:
train_dataset = TensorDataset(X_train, y_train_encoded)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_dataset = TensorDataset(X_test, y_test_encoded)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [55]:
model = nn.Sequential(
    nn.Linear(X_train.shape[1], 8),
    nn.ReLU(),
    nn.Linear(8, num_classes)
)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [57]:
best_f1 = 0.0

for epoch in range(500):
    model.train()  # явно включаем режим обучения перед каждой эпохой
    for batch_X, batch_y in train_loader:
        pred = model(batch_X)
        loss = loss_fn(pred, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 5 == 0:
        model.eval()
        with torch.no_grad():
            logits = model(X_test)
            predictions = torch.argmax(logits, dim=1)
            f1 = f1_score(y_test_encoded.numpy(), predictions.numpy(), average='macro')

        print(f'Эпоха {epoch + 1}, ошибка {loss.item():.4f}, F1 {f1:.3f}')

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), 'best_model.pt')

print(f'Лучший F1: {best_f1:.3f}')

Эпоха 5, ошибка 1.8936, F1 0.246
Эпоха 10, ошибка 1.3142, F1 0.231
Эпоха 15, ошибка 1.6969, F1 0.262
Эпоха 20, ошибка 1.3135, F1 0.240
Эпоха 25, ошибка 1.6387, F1 0.254
Эпоха 30, ошибка 1.7193, F1 0.236
Эпоха 35, ошибка 1.3704, F1 0.228
Эпоха 40, ошибка 1.5396, F1 0.242
Эпоха 45, ошибка 1.6319, F1 0.254
Эпоха 50, ошибка 1.6972, F1 0.251
Эпоха 55, ошибка 1.8245, F1 0.252
Эпоха 60, ошибка 1.4643, F1 0.256
Эпоха 65, ошибка 1.5546, F1 0.242
Эпоха 70, ошибка 1.5459, F1 0.243
Эпоха 75, ошибка 1.4545, F1 0.251
Эпоха 80, ошибка 1.4720, F1 0.247
Эпоха 85, ошибка 1.6329, F1 0.251
Эпоха 90, ошибка 1.8366, F1 0.264
Эпоха 95, ошибка 1.4997, F1 0.247
Эпоха 100, ошибка 1.5846, F1 0.253
Эпоха 105, ошибка 1.7007, F1 0.253
Эпоха 110, ошибка 1.7199, F1 0.244
Эпоха 115, ошибка 1.7743, F1 0.246
Эпоха 120, ошибка 1.6311, F1 0.267
Эпоха 125, ошибка 1.4380, F1 0.272
Эпоха 130, ошибка 1.6276, F1 0.268
Эпоха 135, ошибка 1.3205, F1 0.278
Эпоха 140, ошибка 1.4025, F1 0.256
Эпоха 145, ошибка 1.8497, F1 0.267
Эпоха

In [49]:
model.eval()

with torch.no_grad():
    logits = model(X_test)
    predictions = torch.argmax(logits, dim=1)
    macro_f1 = f1_score(predictions.numpy(), y_test_encoded.numpy(), average='macro')
print(f'F1 на тестовых данных: {macro_f1:.3f}')

F1 на тестовых данных: 0.256
